Elicitation of Consumption and Survey Design
============================================

**Author:** Ethan Ligon



## Practical Difficulties in Measuring Individual Items of Consumption



### The five components of consumption expenditures



### List Length



### Timing & Recall



### Asking about food: GhanaLSS 2016-17



### Asking about food: Uganda 2019-20



### Asking about food: Uganda's single visit



### Asking about non-food: the window stretches



### Asking about non-food: Uganda 2019-20



### Asking about durables: the stock, not the purchase



### Asking about housing: rent, actual and imagined



### Asking about housing: Uganda 2019-20



### Food prices: which price?



### Food prices: what GLSS7 asks, and what it gets



### Food quantities, and what was never bought



### Building the aggregate



`LSMS_Library` does the food side for you. 



#### Preface



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

# The library audits its own corpus on first read and reports what it finds
# --- implausible quantities, NaN index keys, a column that is wholly null in
# one wave --- at multi-paragraph length.  Those reports are a work queue for
# whoever maintains the data, not something the room can act on, and they bury
# the output they are attached to.  Silenced here by their own
# "Set LSMS_..._STRICT=1" signature, which is precise: every other warning,
# pandas deprecations included, still shows.  Delete these two lines to read
# them.
import warnings
warnings.filterwarnings("ignore",
                        message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})

import lsms_library as ll
import numpy as np, pandas as pd

#### Construct data on food expenditures



What features are there?



In [1]:
ll.features()

What countries have `food_acquired`?



In [1]:
ll.countries(feature='food_acquired')

Choosing a country from the list of those that have the `food_acquired` feature:



In [1]:
USE_COUNTRY = 'GhanaLSS'

In [1]:
cntry = ll.Country(USE_COUNTRY)

cntry.food_acquired()

The index is $(t, i, j, u, s, \mathit{visit})$ — wave, household, item,
unit, source, visit — and the columns are `Expenditure`, `Quantity`,
`Price`.  The derived table collapses this to expenditure per household-item:



In [1]:
food = cntry.food_expenditures()
food.head()

Summing over items gives food *purchases* per household.  Not consumption:



In [1]:
hh_fx = food.groupby(['t', 'i', 'v']).sum().squeeze()
hh_fx.groupby('t').describe().round(0)

Note that these purchases always default to being in units of `LCUs` (local currency units).  We can see the units if we ask:



In [1]:
# Total 

hh_fx = cntry.food_expenditures(currency='index').groupby(['t','i','v','currency']).sum()

hh_fx.groupby(['t','currency']).describe().round(0)

Or convert to a common base:



In [1]:
hh_fx = cntry.food_expenditures(numeraire='LCU-real-2017')

In [1]:
hh_fx.groupby(['t','i','v','currency']).sum().groupby(['t','currency']).describe().round(0)

#### Construct data on food out of own production (autoconsumption)



These expenditures don't include autoconsumption (consumption out of own production).  For the GLSS7 those are elicited in a different section:
asks amount and quantity and never a price.
We can include value of all food acquired (included out of production):



In [1]:
fa = cntry.food_acquired()
fa

No value of own production here?  Look at `Expenditure` on the `produced`
rows before you assume it: the library fills it in every wave but 1991-92,
at the household's own reported price, and the `Derivation` column names the
rule that filled it.  Where it is still missing, do the multiplication
yourself.  `min_count=1` keeps a wave that has nothing to sum reported as
missing rather than as a zero.



In [1]:
value = fa['Expenditure'].fillna(fa['Quantity'] * fa['Price'])

by_source = value.groupby(['t', 's']).sum(min_count=1).unstack('s')

by_source['Auto share'] = by_source['produced']/by_source.sum(axis=1)

by_source

Own production is 28.7% of the value of food in 2016-17.  `value` is
expenditure wherever one was recorded or derived, and quantity times the
household's own price where neither was; summed over items it is food
consumption per household, purchased and own-produced, and it is the
aggregate the rest of this session uses.



### GLSS7's answer: a diary, visited six times



### The diary



### GLSS7's six visits



This is the form the diary is kept on, Section 9B: six visit columns headed
*2nd* to *7th*, AMT, QTY and UNIT under each, and nowhere to write a price.
Open it beside the code:
[Section 9B](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Section9B-food-expenditure.pdf) (PDF, opens in a new tab;
also `reading/` in your home directory), and the booklet the household
keeps between visits, which 9B is filled from:
[the diary](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Diary.pdf).
The diary design is visible in the data.  GLSS7 carries a `visit` level:



In [1]:
glss7 = cntry.food_acquired().xs('2016-17', level='t').xs('purchased', level='s')
sorted(glss7.index.get_level_values('visit').unique())

Six of them, five days apart.  The form numbers them 2 to 7, because the
food questions begin at the second visit; the library on the hub labels
them 1 to 6, and other versions use the form's numbers.  The text below uses
the form's numbering, and the code selects the first food visit by position
so that it does not care.  Each visit covers the same five days, so if the
instrument worked perfectly they should all look alike.  They do not:



In [1]:
byvisit = pd.DataFrame({
    'exp':   glss7.groupby(['i', 'visit'])['Expenditure'].sum(),
    'items': glss7.groupby(['i', 'visit']).size(),
})
(byvisit.groupby('visit')
        .agg(mean_exp=('exp', 'mean'), mean_items=('items', 'mean'))
        .assign(exp_per_item=lambda d: d.mean_exp / d.mean_items)
        .round(2))

The first visit stands well above the five that follow, and then the series
is flat.  That is the shape Scott and Amenuvegbe found: steep early, flat
later.  Which of two stories is it, though?  More items remembered, or
bigger amounts attached to the same items?



In [1]:
v1 = sorted(byvisit.index.get_level_values('visit').unique())[0]  # first food visit
first, rest = byvisit.xs(v1, level='visit'), byvisit.drop(v1, level='visit')
for col in ('exp', 'items'):
    print(f"{col:6s} first food visit vs the other five: "
          f"{100 * (first[col].mean() / rest[col].mean() - 1):+.1f}%")

Expenditure is 20.6% higher; the item count only 4.5%.  So it is mostly
larger amounts per item rather than fuller enumeration, which is what
telescoping looks like: purchases from before the window get pulled into
it.  It is the start-up bias Scott and Amenuvegbe had to strip out before
they could measure decay at all, and here it is in the survey you are about
to build an aggregate from.

Whether the first visit is worth keeping is a real question, and
`recall_and_diaries.ipynb` is where you get to argue about it.



### How fast does recall decay?



### What actually drives the error



### Timing: what is a period?



### Inventories: purchases are not consumption



### Measuring consumption despite inventories



### Price Indices



### A household-specific Paasche index



### Durables and housing



## Intra-Household Allocation



### Adult equivalence



### Barten: children re-price goods



### Barten: what the scale depends on



## Constructing Aggregate Welfare Measures



### Poverty measures



### Poverty, correctly weighted



In [1]:
sample = cntry.sample()

def fgt(c, z, alpha=0, w=1):
    """
    Weighted FGT_alpha.  
    c: consumption per adult equivalent; 
    z: poverty line; 
    w: weights.
    """
    # A scalar broadcasts and a Series reindexes, so `w=1' (unweighted) and a
    # real weight column take the same path -- and the weights line up on the
    # index rather than by position.
    w = pd.Series(w, index=c.index, dtype=float)
    # Infinities are not missing values to pandas, so say so before dropping.
    # Dropping on the pair drops a household missing EITHER number, which is
    # what keeps the numerator and denominator over the same households.
    df = (pd.concat({'c': c, 'w': w}, axis=1)
            .replace([np.inf, -np.inf], np.nan)
            .dropna())
    # Sum over the poor only.  Doing it over everybody looks equivalent, and
    # is -- except at alpha=0, where 0**0 == 1 and every household in the
    # country counts as poor.
    poor = df.c < z
    gap = (z - df.c[poor]) / z
    return (df.w[poor] * gap ** alpha).sum() / df.w.sum()

wave = '2016-17'
# food_expenditures is one row per household PER ITEM per source, so it has to
# be summed to the household before any of this means anything: without the
# groupby, `c' is a distribution over individual food purchases, the quartile
# of it is the price of one cheap item, and the headcount counts item-rows
# rather than people.
c = (cntry.food_expenditures(basis='total').squeeze()
          .xs(wave, level='t').groupby('i').sum())
# fgt() aligns on the index itself, but the mean-weight line below does its
# own boolean indexing, and that needs w and c on identical indexes.
w = sample.xs(wave, level='t').weight.groupby('i').first().reindex(c.index)

z = c.quantile(0.25)          # a placeholder line; see the exercise
for a in (0, 1, 2):
    print(f"P_{a} = {fgt(c, z, a, w):.4f}")

print(f"unweighted P_0 = {fgt(c, z):.4f}")
print(f"mean weight, poor = {w[c < z].mean():.3f}"
      f"   non-poor = {w[c >= z].mean():.3f}")

None of which rescues the line itself.  A quartile of the distribution is
not a poverty line: it fixes the unweighted headcount at 0.25 by
construction, in every country and every year, which is a fact about
arithmetic rather than about Ghana.  A real line is
absolute — the cost of a fixed bundle — and does not move with the
distribution, and does not move when everyone gets poorer.  Fixing that is
the first exercise.



### Inequality



### Inequality: Atkinson



### Lorenz and Gini



In [1]:
def lorenz(c, w):
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w) & (c >= 0)
    c, w = c[ok], w[ok]
    order = np.argsort(c)
    c, w = c[order], w[order]
    p = np.cumsum(w) / np.sum(w)
    L = np.cumsum(w * c) / np.sum(w * c)
    return np.concatenate([[0], p]), np.concatenate([[0], L])

def gini(c, w):
    p, L = lorenz(c, w)
    return 1 - 2 * np.trapezoid(L, p)

print(f"Gini (food, purchased and own-produced, {wave}) = {gini(c, w):.3f}")

The Atkinson index needs one more thing than the Gini does, and the code
should make you say it out loud: how averse to inequality you are.



In [1]:
def atkinson(c, w, eps):
    """Weighted Atkinson index.  eps > 0 is inequality aversion."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w) & (c > 0)      # log/power need c > 0
    c, w = c[ok], w[ok]
    mean = np.sum(w * c) / np.sum(w)
    if np.isclose(eps, 1):
        ede = np.exp(np.sum(w * np.log(c)) / np.sum(w))
    else:
        ede = (np.sum(w * c ** (1 - eps)) / np.sum(w)) ** (1 / (1 - eps))
    return 1 - ede / mean

for eps in (0.5, 1.0, 2.0):
    print(f"Atkinson A({eps}) = {atkinson(c, w, eps):.3f}")

The library draws the curve too, and does one thing the code above does
not: it writes every choice that moves the curve into the subtitle.



In [1]:
import matplotlib.pyplot as plt

waves = ['1998-99', '2005-06', '2012-13', '2016-17']

ll.visualizations.lorenz_curve(cntry, waves, per='person', basis='total')
plt.show()

The Gini in the legend is not the one you computed above, and the subtitle
says why: it counts *persons*, weighting each household by its size, where
the code above counted households.  Both are Ginis of food consumption in
the same year.  Neither is wrong; the one you report is a choice, and the chart
makes you state it.

If two Lorenz curves cross, the ranking of the two distributions depends on
which inequality measure you chose, and no scalar will rescue you.  Look
before you summarize.



### One index, or several?



### Exercises



1.  The poverty line used above is fake.  Build a real one: take the food
    bundle consumed by households in the second and third deciles, price it at
    national median unit values, and scale it to 2,900 kcal per adult
    equivalent.  Recompute $P_0, P_1, P_2$.
    Notebook: [`poverty_line.ipynb`](poverty_line.ipynb)
2.  Deflate spatially.  Construct a regional Paasche index from the survey's
    own unit values, apply it, and report how much of the north–south poverty
    gradient survives.
    Notebook: [`spatial_deflation.ipynb`](spatial_deflation.ipynb)
3.  Vary $\theta$ in $c_i = C_i / A_i^\theta$ over $[0.5, 1.0]$ and plot
    the headcount ratio against it.  Over what range does the *ranking* of
    regions change?
    Notebook: [`equivalence_scales.ipynb`](equivalence_scales.ipynb)

